# C2：多车型横向能耗对比
# C2: Cross-Vehicle Energy Comparison

## 一、实验原理 | Principle
### 工程背景 | Engineering Background

新车立项时，工程师做的第一件事是**竞品对标（benchmarking）**：把自家车和 3–5 款竞品放在同一工况、同一口径下横向跑，回答"我们差在哪、差多少"。对标最忌讳各说各话——不同工况、不同测试条件测出的数字不能直接比。本实验把 6 款真实车型放进同一个 UDDS 工况，就是一次标准的对标流程。排名本身不重要，重要的是从排名里读出**设计规律**：什么样的车省电？为什么有的大块头反而省电（5.2 的 Silverado 案例）？

### 1.1 为什么要横向对比？
同一条工况下，不同车型的百公里电耗可能差出 50% 以上。
做车型选型、能量管理策略开发时，第一步就是知道“谁省电、谁费电，以及为什么”。

Why compare? Under the same drive cycle, energy consumption can vary by over 50% across vehicle models.
Vehicle selection and energy-management development always start with knowing who is efficient and why.

### 1.2 能耗的物理来源
电动车跑完一条工况，电池输出的能量去哪了？

Where does the battery energy go over a drive cycle?

- **加速动能** Kinetic energy: `E = ½·m·v²` —— 车越重，加速越费电
- **爬坡势能** Potential energy: `E = m·g·h` —— 有坡度时克服重力做功
- **空气阻力** Aerodynamic drag: `F = ½·ρ·Cd·A·v²` —— 风阻系数 Cd 与迎风面积 A 的乘积决定高速能耗，车速平方关系，高速时主导
- **滚动阻力** Rolling resistance: `F = μ·m·g` —— 与车重成正比，全程都在耗电
- **传动/电机/电池损耗** Drivetrain losses —— 电机效率 map、逆变器、电池内阻
- **制动能量回收** Regenerative braking —— 减速时部分动能回充电池，抵消一部分消耗

### 1.3 FASTSim 2.x 真实仿真
本实验调用 **真实的 FASTSim 2.1.5 引擎**（NREL 开发，Rust 重写的高性能版），
不是查表、不是拟合公式。
每辆车都用 `SimDrive.sim_drive()` 做逐秒动力学仿真：
每个时间步根据工况车速算出需求功率，再经过电机效率 map、电池模型，累积真实电耗。

This lab calls the **real FASTSim 2.1.5 engine** (NREL, Rust rewrite) — no lookup tables, no fitted formulas.
Each vehicle runs a second-by-second `SimDrive.sim_drive()` dynamics simulation:
power demand is computed from the cycle speed trace at every timestep, then passed through
the motor efficiency map and battery model to accumulate true energy use.

### 1.4 工况 UDDS
**UDDS**（Urban Dynamometer Driving Schedule）：美国城市工况，约 12 km，
频繁启停、平均车速低。城市通勤的代表性能耗场景。

**UDDS**: US urban cycle, ~12 km, frequent stop-and-go, low average speed — representative of city commuting.

### 📖 术语速览 | Key Terms

| 术语 Term | 一句话解释 |
|---|---|
| 对标 Benchmarking | 同工况、同口径横向对比竞品，找差距与设计规律 |
| 整备质量 Curb Mass | 空车自重；越重，加速与滚阻耗能越多 |
| 迎风面积 Frontal Area | 车头正投影面积，与 Cd 相乘 (CdA) 决定风阻大小 |
| 平台化 Platform | 多车型共享底盘/三电系统，对标时注意"同平台"车型参数相近 |
| 公告值 Certified Value | 工信部/EPA 公示的续航电耗，用统一工况测得，可与仿真互验 |
## 二、实验目的 | Objectives
1. 掌握 FASTSim 车辆库的批量调用方法（`compare_vehicles`）
2. 在 UDDS 工况下横向对比 6 款真实 BEV 的百公里电耗并排名
3. 分析车重、风阻系数与电耗的关系，理解“重≠费电”的反直觉现象
4. 验证仿真结果的确定性（两次运行排名一致）

1. Learn batch simulation with the FASTSim vehicle library (`compare_vehicles`)
2. Compare 6 real BEVs on the UDDS cycle, rank by kWh/100km
3. Analyze how mass and drag coefficient relate to consumption; understand the counter-intuitive cases
4. Verify simulation determinism (identical ranking across runs)

## 三、学生任务 | Your Tasks
1. **运行**：执行对比代码，得到 6 款车的 kWh/100km 并按能耗从低到高排名
2. **画图**：画柱状图展示排名（注意：matplotlib 里用英文标签，容器没有中文字体）
3. **分析**：最重的车（Silverado EV, ~3849 kg）为什么比 F-150（~2728 kg）更省电？
 提示：对比两者的电机功率和效率工作点
4. **思考题**：
 - Q1：Cupra Born（1927 kg）比 Bolt（~1084 kg）重 78%，电耗却只高 22%，能量去哪了？（提示：制动回收）
 - Q2：如果换成 HWFET 高速工况，你预测排名会怎么变？为什么？动手验证一下
 - Q3：风阻系数 Cd 从 0.44 降到 0.30，在 UDDS 和 HWFET 下分别能省多少电？

1. **Run**: execute the comparison, rank 6 vehicles by kWh/100km (low → high)
2. **Plot**: bar chart of the ranking (use ENGLISH labels in matplotlib — no CJK fonts in container)
3. **Analyze**: why is the heaviest truck (Silverado EV, ~3849 kg) more efficient than the F-150 (~2728 kg)?
 Hint: compare motor power and efficiency operating points
4. **Think**:
 - Q1: Cupra Born (1927 kg) is 78% heavier than Bolt (~1084 kg) but only 22% thirstier — where did the energy go? (Hint: regen)
 - Q2: Predict how the ranking changes on HWFET highway cycle, then verify by running it
 - Q3: How much energy does cutting Cd from 0.44 to 0.30 save on UDDS vs HWFET?

### 🔬 试一试 | Try It

| 改什么 | 怎么改 | 预期效果（先猜后验） |
|---|---|---|
| 工况换高速 | `compare_vehicles(vnums, 'hwfet')` | 排名洗牌：小 Cd 的车名次上升（挑战任务详解） |
| 加入一款皮卡 | 在 vnums 里加一款大型皮卡的编号 | 电耗榜尾大概率被它占据，量级可达小车 2 倍 |
| 只看电耗 < 13 的车 | 对结果表按 `kwh_per_100km` 筛选 | 观察入选车型的共同点：轻？小？低 Cd？ |
## 四、验证内容 | Verification
- [] FASTSim 成功导入（必须用 `Python 3.10 (FASTSim)` 内核！）
- [] 6 款车全部跑出非零能耗
- [] 所有能耗在合理范围（10~25 kWh/100km）
- [] 两次运行排名完全一致（确定性）
- [] 画出了英文标签的排名柱状图

- [] FASTSim imported OK (must use the `Python 3.10 (FASTSim)` kernel!)
- [] All 6 vehicles return non-zero consumption
- [] All values within 10–25 kWh/100km
- [] Two runs produce identical ranking (determinism)
- [] Ranking bar chart with ENGLISH labels

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `No module named 'fastsim'` | 内核选错了，默认内核没有 FASTSim | 右上角切换到 `Python 3.10 (FASTSim)` 内核 |
| `No module named 'common'` | 找不到共享库 | 确认第一行 `sys.path.insert(0, '/home/jovyan/labs')` 已执行 |
| 某车型能耗为 0.0 | 该车型在 FASTSim 2.1.5 车辆库解析异常（电机功率读成 0），不是你的错 | 换一辆车；已知异常编号：33/42/50/53/57/64/68 |
| `KeyError` 或车型不存在 | vnum 编号超出车辆库范围 | 用 `load_bev_list()` 查看可用编号（共 22 款 BEV） |
| 图表中文显示为方框 | 容器没有中文字体 | matplotlib 的标题/坐标轴只用英文 |


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.fastsim_helper import compare_vehicles, load_bev_list
import pandas as pd
import matplotlib.pyplot as plt

# List all available BEVs in the FASTSim vehicle database (real data)
bevs = load_bev_list()
print(f'FASTSim vehicle DB: {len(bevs)} BEVs available')
print('Selected vnums for this lab: [17, 21, 69, 16, 51, 58]')

In [ ]:
# Run REAL FASTSim 2.1.5 SimDrive simulation for 6 diverse BEVs on UDDS
# vnum: 17=Spark EV, 21=Model S60, 69=Silverado EV, 16=Bolt, 51=F-150 Lightning, 58=Cupra Born
vnums = [17, 21, 69, 16, 51, 58]
df = compare_vehicles(vnums, 'udds')

# Attach human-readable names (from FASTSim vehicle database)
names = {
 17: 'Spark EV 2016',
 21: 'Model S60 2016',
 69: 'Silverado EV 2025',
 16: 'Bolt 2017',
 51: 'F-150 Lightning',
 58: 'Cupra Born 2021',
}
df['name'] = df['vnum'].map(names)

# Rank by energy consumption (low -> high)
df_ranked = df.sort_values('kwh_per_100km').reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1
print(df_ranked[['rank', 'name', 'kwh_per_100km']].to_string(index=False))

In [ ]:
# Bar chart of the ranking (ENGLISH labels only - no CJK fonts in container)
plt.figure(figsize=(9, 4.5))
bars = plt.bar(df_ranked['name'], df_ranked['kwh_per_100km'], color='#1a73e8')
plt.ylabel('Energy (kWh/100km)')
plt.title('BEV Energy Ranking on UDDS (FASTSim 2.1.5, real simulation)')
plt.xticks(rotation=20, ha='right')
# Annotate values on bars
for b in bars:
 plt.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.2,
 f'{b.get_height():.1f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

> 📊 **图意说明**：6 款纯电动车在 UDDS 工况下的能耗排名（柱顶为数值）。读图顺序：先看榜首榜尾差几倍，再找 5.2 节的反直觉案例（Silverado EV）在哪根柱，最后想一想：排名和车重排名一致吗？不一致处就是设计（Cd、电驱效率）在起作用。


In [ ]:
# ---- Assertions: real-simulation quality gates ----
# 1. Every vehicle must return non-zero, plausible consumption
for _, row in df_ranked.iterrows():
 e = row['kwh_per_100km']
 assert 10.0 <= e <= 25.0, f"{row['name']}: {e:.1f} kWh/100km out of range!"
print(f'All {len(df_ranked)} vehicles within 10-25 kWh/100km')

# 2. Ranking must be deterministic: re-run and compare order
df2 = compare_vehicles(vnums, 'udds')
rank1 = list(df_ranked['vnum'])
rank2 = list(df2.sort_values('kwh_per_100km')['vnum'])
assert rank1 == rank2, f'Ranking unstable! {rank1} vs {rank2}'
print('Ranking stable across runs:', rank1)
print('PASS')

## 五、结果分析 | Results Analysis

### 5.1 排名解读 | Ranking Interpretation

| 排名 | 车型 | 能耗 (kWh/100km) | 整备质量 (kg) |
|---|---|---|---|
| 1 | Spark EV 2016 | **10.4** | ~1300 |
| 2 | Model S60 2016 | 11.2 | ~2100 |
| 3 | Silverado EV 2025 | 11.3 | ~3850 |
| 4 | Bolt 2017 | 12.5 | ~1620 |
| 5 | F-150 Lightning | 13.5 | ~2730 |
| 6 | Cupra Born 2021 | 15.2 | ~1950 |

**关键发现**：
- 最省电的是最轻的 Spark EV（10.4），符合"轻量化是第一性原理"
- The lightest vehicle (Spark EV) is most efficient — lightweighting is the first principle.

### 5.2 反直觉案例：Silverado EV vs F-150

最重的 Silverado EV（~3850 kg）反而比 F-150（~2730 kg）更省电（11.3 vs 13.5）！
The heaviest Silverado EV outperforms the lighter F-150 — counterintuitive but real.

**原因分析**：
1. **电驱效率**：Silverado 的电机/逆变器效率更高，抵消了重量劣势
2. **能量回收标定**：重车回收策略更激进
3. **教学点**：单一参数（质量）不能决定能耗，系统匹配才是关键

### 5.3 与公告值对比

EPA 采用 5-cycle 加权（含高速/US06），比纯 UDDS 高 20–40% 属正常。相对排名比绝对值更有参考意义。
EPA uses 5-cycle weighting (incl. aggressive US06); 20–40% higher than pure UDDS is normal. Relative ranking matters more than absolute values.

### 5.4 工程启示

1. **选型逻辑**：城市通勤选小车，高速长途看 CdA 而非只看重量
2. **仿真价值**：零成本横向对比，避免"唯参数论"
3. **局限**：FASTSim 为准静态仿真，未计热管理/附件差异
### 5.5 设计改进思考 | Design Improvement Thinking

对标的终点是行动清单。整车工程师拿到这张排名表后通常这样转化：

1. **学榜首的设计**：最省电车型往往三件事都做对了——轻（小电池+轻车身）、低 Cd、电机高效区与巡航点匹配。逐项拆它的参数表。
2. **警惕"堆电池"陷阱**：大电池车续航长但电耗高、成本高、充电慢；对标看电耗（效率）而非只看续航（总量）。
3. **设定自家目标**：把榜首电耗 × 0.95 设为下一代车型目标，再反推质量/Cd/效率的分解指标，分给各科室——这就是整车开发的"目标级联"。
4. **注意口径**：本实验是仿真对标；真实对标还要同台架、同温度、同轮胎复测，仿真负责"先筛方向"，实验负责"最终确认"。


## 挑战任务 | Challenge
1. 把工况换成 `hwfet`（高速）再跑一遍，对比 UDDS 的排名变化，用 `½ρCdAv²` 解释
 Run HWFET instead of UDDS, compare ranking changes, explain with `½ρCdAv²`
2. 从车辆库再挑 2 款车加入对比（避开 33/42/50/53/57/64/68 这些解析异常的编号），看排名如何变化
 Add 2 more vehicles from the DB (avoid broken vnums 33/42/50/53/57/64/68), see how ranking shifts
3. 把本实验最省电的车型参数写入 `labs/data/vehicles/`，供 A1/A2 的 esmini 闭环调用
 [Chain task] Save the most efficient vehicle's params to `labs/data/vehicles/` for A1/A2 esmini closed-loop use


---
## 阶段五：结果文件归档与平台官方评测 | Auto-Grading & Results Sync


In [ ]:
# 阶段五：结果文件归档与平台官方评测
import json, os
from common.evaluator import grade_metrics

record = {
    'experiment': 'C2_vehicle_comparison',
    'metrics': {
        'n_vehicles': int(len(vehicles))
    }
}
saved_ok = False
for target_dir in ['/home/jovyan/labs/data/results', 'labs/data/results', 'results']:
    try:
        os.makedirs(target_dir, exist_ok=True)
        out_file = os.path.join(target_dir, 'C2_vehicle_comparison.json')
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f'✓ 结果文件已成功归档至: {out_file}')
        saved_ok = True
        break
    except OSError:
        continue
eval_report = grade_metrics('C2_vehicle_comparison', record['metrics'])
print(f'🏆 官方评分: {eval_report["score"]} / 100 分 ({'✅ 通过' if eval_report["passed"] else '❌ 未通过'})')


---

## 下一个实验 | Next →

**[C3_sensitivity_analysis](./C3_sensitivity_analysis.ipynb)**

> **所需内核**：`py310-fastsim` | **Required kernel**: `py310-fastsim`
